In [1]:
from pathlib import Path
import matplotlib.pyplot as plt
import time
from import_xy import parse_xy_file
%matplotlib widget


# SPECS Prodigy `.xy` file structure

A `.xy` export has a four-level hierarchy:

```
File
 └─ Group     ("# Group: <name>") — a measurement session/context (Scan Mode, lens
               voltage, calibration file, source parameters, ...). A new Group block
               is written every time the operator switches context in SpecsLab Prodigy.
     └─ Region  ("# Region: <name>") — one measured spectrum window, e.g. O1s, C1s, VB.
                 Every Region belongs to whichever Group precedes it.
         └─ Scan  ("# Cycle/Scan/Channel") — one energy/intensity trace. A Region
                    can hold several Scans (repeats, or points of a positional map).
```

A Region's `kind` follows from what's inside it:
- **`single`** — one Scan, no Parameters → a one-off spectrum.
- **`series`** — more than one Scan, no Parameters → repeat scans of the same acquisition (averaging or time-tracking).
- **`map`** — Scans carry Parameters (Step/Y/Z position) → position-resolved (2D) data.

The cells below load one `.xy` file and print out, level by level: the Groups (with their metadata), the Regions (with their kind), the file-level metadata, and finally a plot of the spectra.

**Load the file, then list its Groups.** Each `elem.name` is a measurement-session label; the printed metadata (Scan Mode, Analyzer Lens Voltage, Polar/Azimuth Angle, `ex_energy`, ...) applies to every Region recorded under that Group, until the next Group block appears.

In [2]:
data_path = Path('data/NormalXPS.xy')
t_start = time.time()
xy = parse_xy_file(data_path)
t_end = time.time()
print(f"Data loaded in {t_end - t_start:.2f} seconds")

# print the names of the measurement groups and their corresponding metadata
for elem in xy.groups:
    print(elem.name)
    for e in elem.metadata.items():
        print(e)

Data loaded in 0.01 seconds
Cell vacuum 50C
('Scan Mode', 'FixedAnalyzerTransmission')
('Analyzer Lens Voltage', '1.5kV')
('Calibration File', 'C:/Users/Public/Documents/SpecsLab Prodigy/Database/DatasetCalib1D\\latest-fat.calib1d')
('Transmission File', 'C:\\Program Files\\SPECS\\SpecsLab Prodigy\\database\\TransmissionFunction/Default/Default.tf')
('Analyzer Slit', '3:1x20\\B:open')
('Iris Diameter', 60)
('Name', 'Monochromator')
('Polar Angle', '45 °')
('Azimuth Angle', '45 °')
('ex_energy', 49.99794762701864)
('device_state', 'operating')


In [ ]:
# print the 'regions' in the file
xy.regions

In [ ]:
# print the file metadata:
for elem in xy.file_metadata.items():
    print(elem)

In [ ]:
# plot all the XPS data as a function of binding energy

num_regions = len(xy.regions)
num_regions = num_regions if num_regions <= 10 else 10 # used to limit the amount of data if the file contains many scans

plt.close()
fig, ax = plt.subplots()
for r in xy.regions[:num_regions]:
    #print(r)
    for s in r.scans[:num_regions]:
        ax.xaxis.set_inverted(True)
        if len(s.intensity[:].shape) <= 1:
            ax.plot(s.energy[:], s.intensity[:], label=r.name)

ax.set_xlabel('Binding Energy (eV)')
ax.set_ylabel('Intensity (counts)')
ax.legend()